# 🧠 Özellik Üretimi (Feature Creation / Engineering)

Modelin performansını artırmak için var olan verilerden matematiksel veya mantıksal olarak yeni sütunlar (feature'lar) üretme şablonları.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

## 1. Domain Knowledge (Alan Bilgisi) ile Üretim
Verinin hikayesini okuyarak mantıksal sütunlar yaratmak Kaggle'da en çok kazandıran yöntemdir.

In [ ]:
def create_domain_features(df):
    """
    House Prices veya Titanic gibi projelere göre uyarlanması gereken mantıksal çıkarımlar.
    """
    df_new = df.copy()
    
    # Örnek 1: İki alanı toplayarak toplam alanı bulmak (House Prices)
    if '1stFlrSF' in df_new.columns and '2ndFlrSF' in df_new.columns:
        df_new['Total_Home_Quality'] = df_new['OverallQual'] + df_new['OverallCond']
        df_new['Total_SF'] = df_new['1stFlrSF'] + df_new['2ndFlrSF'] + df_new.get('TotalBsmtSF', 0)
        
    # Örnek 2: Spaceship Titanic için toplam harcama
    expense_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    if all(col in df_new.columns for col in expense_cols):
        df_new['Total_Expense'] = df_new[expense_cols].sum(axis=1)
        df_new['Is_Spender'] = (df_new['Total_Expense'] > 0).astype(int)
        
    return df_new

## 2. Binning (Gruplama)
Sürekli (continuous) sayısal değişkenleri yaş grupları, fiyat dilimleri gibi kategorilere ayırmak.

In [ ]:
def apply_binning(df, column, q=4, labels=None):
    """
    qcut: Veriyi yüzdelik dilimlere (çeyreklikler gibi) eşit sayıda eleman düşecek şekilde böler.
    cut: Veriyi değer aralıklarına göre eşit aralıklara böler.
    """
    df_new = df.copy()
    
    # qcut örneği (çeyreklikler - %25, %50, %75)
    new_col_name = f"{column}_Binned"
    if labels:
        df_new[new_col_name] = pd.qcut(df_new[column], q=q, labels=labels)
    else:
        df_new[new_col_name] = pd.qcut(df_new[column], q=q)
        
    return df_new

# Örnek:
# df_train = apply_binning(df_train, 'Age', q=4, labels=['Child', 'Youth', 'Adult', 'Senior'])

## 3. Polynomial Features (Polinomsal Çoğaltma)
Mevcut sayısal sütunların karelerini, küplerini veya birbirleriyle çarpımlarını otomatik olarak üretir.
Lineer modellerin karmaşık ilişkileri öğrenmesini sağlar.

In [ ]:
def create_polynomial_features(df_train, df_test, cols_to_poly, degree=2):
    """
    Sadece belirli sütunları (cols_to_poly) seçip polinomsal özelliklerini çıkarır ve orijinal veriye ekler.
    """
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    
    # Train için fit ve transform
    poly_train = poly.fit_transform(df_train[cols_to_poly])
    poly_test = poly.transform(df_test[cols_to_poly])
    
    # Çıktı bir Numpy dizisidir, onu tekrar DataFrame'e çevirelim
    feature_names = poly.get_feature_names_out(cols_to_poly)
    
    df_poly_train = pd.DataFrame(poly_train, columns=feature_names, index=df_train.index)
    df_poly_test = pd.DataFrame(poly_test, columns=feature_names, index=df_test.index)
    
    # Orijinal dataframe'deki poly kolonlarını düşür ve yeni üretilenleri ekle (Duplicate olmasın diye)
    df_train_final = pd.concat([df_train.drop(columns=cols_to_poly), df_poly_train], axis=1)
    df_test_final = pd.concat([df_test.drop(columns=cols_to_poly), df_poly_test], axis=1)
    
    return df_train_final, df_test_final